# Problem Book P1, Chapter 3: Primes, Seen from Everywhere (Python companion)

Part A checks every number of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math, random, time
from math import comb, gcd
from fractions import Fraction as Fr
from functools import lru_cache
import numpy as np
import sympy as sp
from sympy import isprime, factorint, primerange, totient, n_order, legendre_symbol, jacobi_symbol
import matplotlib.pyplot as plt
random.seed(3)

## Part A: checks

**Problem 3.1 (Fermat numbers).** The prime factors of $F_5$ and $F_6$ are $\equiv1\pmod{2^{n+2}}$, Euler's $641$ divides $F_5$,
and the first Fermat numbers are pairwise coprime.

In [ ]:
F = lambda n: 2**(2**n) + 1
for n in [5, 6]:
    fs = factorint(F(n))
    print(n, fs, all(q % 2**(n + 2) == 1 for q in fs))
print(641 == 5 * 2**7 + 1 == 5**4 + 2**4, F(5) % 641)
print(all(gcd(F(m), F(n)) == 1 for m in range(9) for n in range(m)), math.prod(F(k) for k in range(6)) == F(6) - 2)

**Problem 3.2 (Mersenne divisors).** Prime factors of $M_p$ are $\equiv1\pmod{2p}$ and $\equiv\pm1\pmod8$; and $2p+1\mid M_p$ for
$p=11,23,83$.

In [ ]:
ok = True
for p in primerange(3, 70):
    for q in factorint(2**p - 1):
        ok &= q % (2 * p) == 1 and q % 8 in (1, 7)
print(ok, [(p, 2 * p + 1, (2**p - 1) % (2 * p + 1)) for p in [11, 23, 83]])

**Problem 3.3 (pseudoprimes).** $341$, the chain $n\mapsto2^n-1$, and the family $\frac{4^p-1}3$.

In [ ]:
def psp(n):
    return n % 2 == 1 and not isprime(n) and pow(2, n - 1, n) == 1
m = 2**341 - 1
print(psp(341), psp(m))
print([(p, psp((4**p - 1) // 3)) for p in primerange(5, 60)])
print("smallest base-2 pseudoprimes:", [n for n in range(3, 3000, 2) if psp(n)])

**Problem 3.4 (Carmichael numbers).** Korselt's criterion against the definition, the Chernick family, and the smallest
examples.

In [ ]:
def carm_def(n):
    return n > 2 and not isprime(n) and all(pow(a, n - 1, n) == 1 for a in range(2, n) if gcd(a, n) == 1)
def korselt(n):
    f = factorint(n)
    return n > 2 and not isprime(n) and all(e == 1 for e in f.values()) and all((n - 1) % (p - 1) == 0 for p in f)
C = [n for n in range(3, 20000) if korselt(n)]
print(C, all(carm_def(n) for n in C[:6]), all(len(factorint(n)) >= 3 and n % 2 for n in C))
ch = [k for k in range(1, 60) if isprime(6*k + 1) and isprime(12*k + 1) and isprime(18*k + 1)]
print(ch[:6], (6 + 1) * (12 + 1) * (18 + 1), all(korselt((6*k + 1) * (12*k + 1) * (18*k + 1)) for k in ch))

**Problem 3.5 (Euler liars).** For every odd composite $n<1500$, the Euler liars are at most half of the units, and they are
closed under multiplication.

In [ ]:
def euler_liars(n):
    return [a for a in range(1, n) if gcd(a, n) == 1 and pow(a, (n - 1) // 2, n) == jacobi_symbol(a, n) % n]
worst = 0
for n in range(9, 1500, 2):
    if isprime(n): continue
    L = euler_liars(n)
    worst = max(worst, Fr(len(L), int(totient(n))))
print("largest proportion:", worst)
L = set(euler_liars(1729)); print(len(L), int(totient(1729)), all(a * b % 1729 in L for a in list(L)[:40] for b in list(L)[:40]))

**Problem 3.6 (the AKS congruence).** $(X+a)^n\equiv X^n+a\pmod n$ holds for primes and fails for composites (here $a=1$ and
$a=2$, $n<200$).

In [ ]:
def aks_holds(n, a):
    return all(comb(n, k) * pow(a, n - k, n) % n == 0 for k in range(1, n)) and (pow(a, n, n) - a) % n == 0
print(all(aks_holds(n, a) == isprime(n) for n in range(2, 200) for a in [1, 2] if gcd(a, n) == 1))
print("Carmichael 561 with a=2:", aks_holds(561, 2))

**Problem 3.7 (Pratt certificates).** The criterion of (a), a certificate tree, and the bound $f(p)\le2\log_2p-1$.

In [ ]:
def lucas_crit(n):
    qs = factorint(n - 1)
    return any(pow(a, n - 1, n) == 1 and all(pow(a, (n - 1) // q, n) != 1 for q in qs) for a in range(2, n))
print(all(lucas_crit(n) == isprime(n) for n in range(3, 1500)))
@lru_cache(None)
def f(p):
    return 1 if p == 2 else 1 + sum(f(q) for q in factorint(p - 1))
P = list(primerange(3, 100000))
print(all(f(p) <= 2 * math.log2(p) - 1 for p in P), max(f(p) / math.log2(p) for p in P))
def tree(p, depth=0):
    a = sp.primitive_root(p) if p > 2 else 1
    print("  " * depth + f"{p}: a={a}, p-1 = {dict(factorint(p - 1)) if p > 2 else '-'}")
    for q in (factorint(p - 1) if p > 2 else []):
        if q > 2: tree(q, depth + 1)
tree(1000003)

**Problem 3.8 (Pocklington and Proth).** Proth's test for $3329$ with $a=3$, and Proth's test against `isprime` for small
$k2^m+1$.

In [ ]:
print(3329 == 13 * 2**8 + 1, pow(3, 3328 // 2, 3329) == 3329 - 1)
def proth(N):
    return any(pow(a, (N - 1) // 2, N) == N - 1 for a in range(2, 60))
cases = [(k << m) + 1 for m in range(1, 14) for k in range(1, 2**m, 2)]
print(len(cases), all(proth(N) == isprime(N) for N in cases))

**Problem 3.9 (Monier's formula).** Strong liars counted by brute force agree with the formula for every odd composite
$n<3000$; the largest proportions are $\frac13$ at $9$ and $\frac14$ at $15$, $91$, $703$, $1891$.

In [ ]:
def strong_liars(n):
    s, d = 0, n - 1
    while d % 2 == 0: d //= 2; s += 1
    c = 0
    for a in range(1, n):
        if gcd(a, n) != 1: continue
        x = pow(a, d, n)
        if x == 1 or x == n - 1: c += 1; continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1: c += 1; break
    return c
def monier(n):
    f = factorint(n); r = len(f)
    d = n - 1
    while d % 2 == 0: d //= 2
    g, nus = 1, []
    for p in f:
        m, nu = p - 1, 0
        while m % 2 == 0: m //= 2; nu += 1
        nus.append(nu); g *= gcd(d, m)
    nu = min(nus)
    return g * (1 + (2**(r * nu) - 1) // (2**r - 1))
comp = [n for n in range(9, 3000, 2) if not isprime(n)]
print(all(strong_liars(n) == monier(n) for n in comp))
best = sorted(((Fr(monier(n), int(totient(n))), n) for n in range(9, 6000, 2) if not isprime(n)), reverse=True)
print(best[:6])

**Problem 3.10 (Lucas--Lehmer).** The test against `isprime` for $p<62$, and the Mersenne primes it finds.

In [ ]:
def lucas_lehmer(p):
    M, s = 2**p - 1, 4
    for _ in range(p - 2): s = (s * s - 2) % M
    return s == 0
print(all(lucas_lehmer(p) == isprime(2**p - 1) for p in primerange(3, 62)))
print([p for p in primerange(3, 130) if lucas_lehmer(p)])

**Problems 3.11 and 3.12 (Wolstenholme, Jacobsthal).** The harmonic sums modulo $p$ and $p^2$, the binomial modulo $p^3$, and
$\binom{ap}{bp}\equiv\binom ab$ modulo $p^2$ (and modulo $p^3$ for $p\ge5$). The only $p<20000$ with Wolstenholme modulo $p^4$
are printed.

In [ ]:
def H(p, r=1):
    return sum(Fr(1, k**r) for k in range(1, p))
print(all(H(p, 2).numerator % p == 0 and H(p).numerator % p**2 == 0 for p in primerange(5, 120)))
print(all(comb(2*p - 1, p - 1) % p**3 == 1 for p in primerange(5, 400)))
print([p for p in primerange(5, 20000) if comb(2*p - 1, p - 1) % p**4 == 1])
print(all(comb(a*p, b*p) % p**2 == comb(a, b) % p**2 for p in primerange(2, 30) for a in range(1, 7) for b in range(a + 1)),
      all(comb(a*p, b*p) % p**3 == comb(a, b) % p**3 for p in primerange(5, 30) for a in range(1, 7) for b in range(a + 1)))

**Problem 3.13 (Fermat quotients).** The logarithm rules, the alternating harmonic sum, Lerch's formula, and the Wieferich
primes below $10^5$.

In [ ]:
def q(a, p): return ((pow(a, p - 1, p * p) - 1) // p) % p
ok = True
for p in primerange(3, 200):
    inv = lambda k: pow(k, -1, p)
    for a in range(1, 12):
        for b in range(1, 12):
            if a % p and b % p:
                ok &= q(a * b, p) == (q(a, p) + q(b, p)) % p
                ok &= q(a + p * b, p) == (q(a, p) - b * inv(a)) % p
    ok &= 2 * q(2, p) % p == sum((-1)**(k - 1) * inv(k) for k in range(1, p)) % p
    W = ((math.factorial(p - 1) + 1) // p) % p
    ok &= sum(q(a, p) for a in range(1, p)) % p == W
print(ok)
wief = [p for p in primerange(3, 10**5) if pow(2, p - 1, p * p) == 1]
print(wief, [(p, n_order(2, p), n_order(2, p * p)) for p in wief + [7, 11]])
print("Wilson primes below 1000:", [p for p in primerange(2, 1000) if (math.factorial(p - 1) + 1) % (p * p) == 0])

**Problems 3.14 and 3.15 (Morley, Glaisher).** Morley's congruence modulo $p^2$, and modulo $p^3$ for $p\ge5$; Glaisher's
$(p-1)!\equiv pB_{p-1}-p\pmod{p^2}$.

In [ ]:
def red(fr, m):
    fr = sp.Rational(fr)
    return int(fr.p) * pow(int(fr.q), -1, m) % m
ok2 = ok3 = okg = True
for p in primerange(3, 200):
    k = (p - 1) // 2
    ok2 &= comb(p - 1, k) % p**2 == (-1)**k * pow(4, p - 1, p**2) % p**2
    if p >= 5: ok3 &= comb(p - 1, k) % p**3 == (-1)**k * pow(4, p - 1, p**3) % p**3
    pB = p * sp.bernoulli(p - 1)
    okg &= sp.Rational(pB).q % p != 0 and math.factorial(p - 1) % p**2 == red(pB - p, p**2)
print(ok2, ok3, okg)

**Problem 3.16 (Gaussian primes).** The count $G(x)$ by brute force against the formula, and the ratio $G(x)\log x/(4x)$.

In [ ]:
def G_brute(x):
    R, c = int(math.isqrt(x)) + 1, 0
    for a in range(-R, R + 1):
        for b in range(-R, R + 1):
            N = a * a + b * b
            if 0 < N <= x:
                if a == 0 or b == 0:
                    m = abs(a + b); c += isprime(m) and m % 4 == 3
                else:
                    c += isprime(N)
    return c
def G_formula(x):
    p1 = sum(1 for p in primerange(2, x + 1) if p % 4 == 1)
    p3 = sum(1 for p in primerange(2, math.isqrt(x) + 1) if p % 4 == 3)
    return 4 * (1 + 2 * p1 + p3)
for x in [100, 1000, 10000]:
    print(x, G_brute(x), G_formula(x))
for x in [10**4, 10**5, 10**6, 10**7]:
    print(x, G_formula(x) * math.log(x) / (4 * x))

The ratio creeps down to $1$ as slowly as $\pi(x)\log x/x$ does: the same second-order term $1+\frac1{\log x}$.

**Problem 3.17 (Euclid--Mullin).** The first sequence (smallest factor) and the second one (largest factor), where $5$ never
shows up.

In [ ]:
for pick in [min, max]:
    P, seq = 1, []
    for i in range(9):
        a = pick(factorint(P + 1)); seq.append(a); P *= a
    print(pick.__name__, seq)

**Problem 3.18 ($n^2+1$).** The largest prime factor of $\prod_{n\le x}(n^2+1)$, which beats $x$ by a wide margin.

In [ ]:
for x in [10, 100, 1000, 3000]:
    Px = max(max(factorint(n * n + 1)) for n in range(1, x + 1))
    print(x, Px, Px / x)

**Problem 3.19 (Nair).** $m\binom nm\mid d_n$, $d_n\ge2^n$ from $n=7$ on, and the lower bound for $\pi(n)$.

In [ ]:
d = 1; ok = True
for n in range(1, 200):
    d = d * n // gcd(d, n)
    ok &= all(d % (m * comb(n, m)) == 0 for m in range(1, n + 1))
    if n >= 7: ok &= d >= 2**n and sp.primepi(n) >= n * math.log(2) / math.log(n)
print(ok, [n for n in range(1, 8) if math.lcm(*range(1, n + 1)) < 2**n])

**Problem 3.20 (safe primes).** $\pm2$ as primitive roots for the safe primes $p=2q+1$.

In [ ]:
ok = True
for qq in primerange(5, 3000):
    p = 2 * qq + 1
    if isprime(p) and p > 7:
        g = 2 if qq % 4 == 1 else p - 2
        ok &= n_order(g, p) == p - 1
print(ok)

**Problem 3.21 (finding a prime).** Expected draws $\log N$ and $\frac4{15}\log N$ for $N=2^{1023}$, and Mertens's saving
$\prod_{p\le y}(1-\frac1p)\approx\frac{e^{-\gamma}}{\log y}$. Then a simulation at $N=2^{64}$.

In [ ]:
L = 1023 * math.log(2); print(round(L), round(4 * L / 15))
for y in [5, 100, 10**4]:
    print(y, math.prod(1 - 1 / p for p in primerange(2, y + 1)), math.exp(-0.5772156649) / math.log(y))
N = 2**64; draws = []
for _ in range(2000):
    k = 0
    while True:
        k += 1; n = random.randrange(N, 2 * N)
        if isprime(n): break
    draws.append(k)
print("mean draws:", np.mean(draws), "prediction:", math.log(N))

**Problem 3.22 (least non-residue).** $n_p$ is prime and $<\sqrt p+1$; the model constant $\sum p_k2^{-k}$; the true average.

In [ ]:
def least_qnr(p):
    a = 2
    while pow(a, (p - 1) // 2, p) == 1: a += 1
    return a
P = list(primerange(3, 200000))
nps = [least_qnr(p) for p in P]
print(all(isprime(n) and n < math.sqrt(p) + 1 for n, p in zip(nps, P)))
pr = list(primerange(2, 400)); print("model:", sum(pr[k] / 2**(k + 1) for k in range(len(pr))))
print("average over p < 2*10^5:", np.mean(nps))

## Part B: laboratory

**L3.1 (a census of liars).** For odd composites $n<2\cdot10^5$, use Monier's formula to compute the proportion of strong
liars. What fraction of these $n$ have *only* the trivial liars $\pm1$? Which $n$ have proportion above $\frac18$? Guess their
shape.

**L3.2 (hunting Mersenne primes).** Run Lucas--Lehmer for every prime $p<2300$. Which $M_p$ are prime? Time the test as a
function of $p$, and fit the exponent of $p$ in the running time. Before running a test, sieve $M_p$ by the divisors $2kp+1$
of Problem 3.2: how many exponents does trial division up to $10^6$ eliminate?

**L3.3 (are Fermat quotients random?).** For primes $p<10^6$, compute $u_p=q_p(2)/p\in[0,1)$. Draw the histogram. If the $u_p$
were uniform, a prime would be Wieferich with probability $\frac1p$, and the expected count up to $x$ would be about $\log\log x$.
Compare with the count you find.

**L3.4 (records of the least non-residue).** For $p<10^6$, list the primes $p$ where $n_p$ beats all earlier values. Compare
the record values with $\log p\cdot\log\log p$ and with the bound $2\log^2p$ that the generalised Riemann hypothesis gives
(Problem 3.22). Which looks closer?

## Solutions of the laboratory

### L3.1

In [ ]:
props, trivial, big = [], 0, []
for n in range(9, 200000, 2):
    if isprime(n): continue
    L = monier(n); ph = int(totient(n))
    props.append(L / ph); trivial += (L == 2)
    if 8 * L > ph: big.append((n, dict(factorint(n))))
print(len(props), "composites;", trivial / len(props), "have only the liars +1 and -1")
print(big)
plt.figure(figsize=(7, 3)); plt.hist(np.log10(props), bins=60); plt.xlabel("log10(proportion of strong liars)"); plt.show()

Almost two thirds of odd composites have no strong liars except $\pm1$, and the typical proportion is tiny. The few with
proportion above $\frac18$ are the squares $9,25,49$, a handful of Carmichael numbers such as $8911$ and $29341$, and above all
products $pq$ of two primes where $q-1$ is a small multiple of $p-1$: $q=2p-1$ as in $91=7\cdot13$, $703=19\cdot37$,
$1891=31\cdot61$ (these attain the bound $\frac14$ of Problem 3.9, like $15=3\cdot5$), or $q-1=3(p-1)$ as in $21=3\cdot7$,
$133=7\cdot19$, $341=11\cdot31$. Monier's formula explains it: such $n$ make $\gcd(d,m_i)$ large for both factors. One random base already catches almost every composite; the $4^{-k}$ bound is for the worst case.

### L3.2

In [ ]:
def small_factor(p, bound=10**6):
    k = 1
    while 2 * k * p + 1 <= bound:
        qq = 2 * k * p + 1
        if qq % 8 in (1, 7) and pow(2, p, qq) == 1: return qq
        k += 1
    return None
ps = list(primerange(3, 2300))
sieved = [p for p in ps if small_factor(p) and small_factor(p) < 2**p - 1]
print(len(ps), "exponents,", len(sieved), "eliminated by trial division")
found, times = [], []
for p in ps:
    t = time.perf_counter(); r = lucas_lehmer(p); times.append((p, time.perf_counter() - t))
    if r: found.append(p)
print(found)
big = [(p, t) for p, t in times if p > 500]
slope = np.polyfit(np.log([p for p, _ in big]), np.log([t for _, t in big]), 1)[0]
print("running time ~ p^%.2f" % slope)

The exponents found are $3,5,7,13,17,19,31,61,89,107,127$ and then $521,607,1279,2203,2281$, the five that Robinson found in
1952 on the SWAC, the first computer search. Trial division removes about $40\%$ of the exponents before any squaring. The test
does $p$ squarings of $p$-bit numbers, so the time grows like $p^{1+\alpha}$ where $p^\alpha$ is the cost of one squaring and one
reduction: with Python's Karatsuba arithmetic, $\alpha\approx1.6$ in theory, and the fit gives an exponent near $2.7$ (it depends
on the machine). GIMPS replaces squaring by an FFT, with cost $p\log p$, and the
test time becomes about $p^2\log p$.

### L3.3

In [ ]:
P = list(primerange(3, 10**6))
u = np.array([q(2, p) / p for p in P])
plt.figure(figsize=(7, 3)); plt.hist(u, bins=50); plt.xlabel("q_p(2)/p"); plt.show()
print("mean", u.mean(), "Wieferich primes:", [p for p in P if q(2, p) == 0], "loglog(10^6) =", math.log(math.log(10**6)))

The histogram is flat: as far as anyone can see, $q_p(2)$ behaves like a random residue modulo $p$. The model predicts
$\sum_{p\le x}\frac1p\approx\log\log x+0.26\approx2.9$ Wieferich primes below $10^6$, and there are $2$: $1093$ and $3511$.
Dorais and Klyve (2011) searched up to $6.7\cdot10^{15}$ without finding a third. Between $10^6$ and there the model expects
$\log\log(6.7\cdot10^{15})-\log\log10^6\approx1$ more, so the silence is mildly surprising, and still consistent with chance.

### L3.4

In [ ]:
rec, best = [], 0
for p in primerange(3, 10**6):
    n = least_qnr(p)
    if n > best: best = n; rec.append((p, n))
for p, n in rec:
    print(p, n, round(math.log(p) * math.log(math.log(p)), 1) if p > 3 else "-", round(2 * math.log(p)**2))

The records grow very slowly, and they track $\log p\log\log p$ closely (from $p=71$ on, the ratio stays between $1$ and $1.35$), far below
$2\log^2p$. This is the conjecture: $n_p$ is as large as a constant times $\log p\log\log p$ infinitely often (Montgomery proved
this assuming the Riemann hypothesis for $L$-functions) and never much larger. Without any hypothesis, the proofs are stuck at
Burgess's power $p^{0.152}$.